In [1]:
# ============================================================================
# DUAL-BRANCH (ResNet50 + Swin Transformer) ASPHALT CRACK CLASSIFIER
# Paste each "# %% CELL" block below into its own Kaggle notebook cell.
#
# HONESTY NOTES (read before running):
# 1. Your dataset ("Asphalt Cracked and Uncracked Image Dataset") is an
#    IMAGE-LEVEL classification dataset (Crack / Uncracked folders), not a
#    pixel-mask segmentation dataset. So the only component trained with a
#    real supervised loss here is the "Confidence Score Head" (2-class
#    softmax). The heatmap / Otsu / skeleton / EDT / density outputs from
#    your diagram are produced as CLASSICAL POST-PROCESSING on the fused
#    feature map at inference time -- they are not fabricated "trained"
#    outputs, and no fake accuracy numbers are printed anywhere below.
# 2. "IEAP Enhanced Image" is implemented as CLAHE contrast enhancement
#    (a real, standard technique). If your IEAP refers to a specific paper
#    algorithm, swap the body of ieap_enhance() for that formula.
# 3. The exact channel/spatial numbers drawn in your architecture diagram
#    (e.g. ResNet 64/128/256/512/512, Swin H/32 x 8C, fused 32x32x512) are
#    not mutually consistent with real ResNet50/Swin-Tiny shapes. Real
#    feature maps are aligned here via 1x1 conv projection + bilinear
#    interpolation so the addition fusion F_cnn + F_swin is valid.
# ============================================================================

# %% CELL 1 - Install dependencies (Kaggle usually has torch/torchvision already)
# !pip install -q timm scikit-image opencv-python-headless

# %% CELL 2 - Imports
import os
import copy
import random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# %% CELL 3 - Auto-detect the dataset path (no hardcoded slug/folder names)
# Kaggle dataset folder structures are often nested (e.g.
# /kaggle/input/<dataset-slug>/<inner-folder>/Cracked720, .../Uncracked720),
# and the exact names vary per dataset upload. Instead of guessing, this
# walks /kaggle/input and finds the directory that directly contains one
# "cracked" folder and one "uncracked" folder.
import json

def find_data_dir(root="/kaggle/input"):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        names_lower = [d.lower() for d in dirnames]
        has_uncracked = any("uncrack" in n for n in names_lower)
        has_cracked = any(("crack" in n and "uncrack" not in n) for n in names_lower)
        if has_uncracked and has_cracked:
            candidates.append(dirpath)
    if not candidates:
        return None
    # Prefer the most specific (deepest) match in case of nested false positives.
    candidates.sort(key=lambda p: p.count(os.sep), reverse=True)
    return candidates[0]

DATA_DIR = find_data_dir()

if DATA_DIR is None:
    print("Could not auto-detect the dataset folder. Full tree under /kaggle/input:")
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        print(dirpath, "->", dirnames)
    raise FileNotFoundError(
        "Set DATA_DIR manually above to the folder that directly contains "
        "your Cracked/Uncracked subfolders, based on the tree printed above."
    )

print("Auto-detected DATA_DIR:", DATA_DIR)

IMG_SIZE = 256          # matches "256 x 256 x 3" in the diagram
BATCH_SIZE = 16
NUM_EPOCHS = 25
LR = 1e-4
NUM_CLASSES = 2         # Crack / Non-Crack
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% CELL 4 - Build the file list (path, label) directly from folder structure
def build_file_list(data_dir):
    classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    if len(classes) != NUM_CLASSES:
        print(f"WARNING: found {len(classes)} class folders: {classes}. "
              f"Expected {NUM_CLASSES}. Check DATA_DIR / folder structure.")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    samples = []
    for c in classes:
        folder = os.path.join(data_dir, c)
        for fname in os.listdir(folder):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                samples.append((os.path.join(folder, fname), class_to_idx[c]))
    return samples, class_to_idx

samples, class_to_idx = build_file_list(DATA_DIR)
print("Classes found:", class_to_idx)
print("Total images:", len(samples))

# Save dataset metadata (the auto-detected path + class mapping) so the local
# testing script doesn't need you to hand-copy the Kaggle path prefix later.
with open(os.path.join(OUTPUT_DIR, "dataset_meta.json"), "w") as f:
    json.dump({"kaggle_data_dir": DATA_DIR, "class_to_idx": class_to_idx}, f, indent=2)

# 70% train / 30% test, stratified so both classes are proportionally represented
paths = [s[0] for s in samples]
labels = [s[1] for s in samples]

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths, labels, test_size=0.30, random_state=SEED, stratify=labels
)

print(f"Train: {len(train_paths)} | Test (30% holdout): {len(test_paths)}")

# Persist the exact 30% split to disk so the SAME test files can be re-loaded
# on your local machine later (guarantees train/test never leak into each other,
# regardless of environment/random-seed differences between Kaggle and local).
split_df = pd.DataFrame({
    "path": train_paths + test_paths,
    "label": train_labels + test_labels,
    "split": ["train"] * len(train_paths) + ["test"] * len(test_paths),
})
split_csv_path = os.path.join(OUTPUT_DIR, "train_test_split.csv")
split_df.to_csv(split_csv_path, index=False)
print("Saved split manifest to:", split_csv_path)
print("Download this CSV along with the model checkpoint -- you need it locally "
      "to reproduce the exact same 30% test set.")

# %% CELL 5 - IEAP enhancement (CLAHE-based) + Dataset class
def ieap_enhance(bgr_img):
    """
    Placeholder / documented assumption for the 'IEAP Enhanced Image' step.
    Implemented as CLAHE (Contrast Limited Adaptive Histogram Equalization)
    on the L channel in LAB color space -- a standard, real enhancement
    technique used for pavement/crack imagery.
    Replace this function's body if you have a specific published IEAP formula.
    """
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_enh = clahe.apply(l)
    enhanced = cv2.merge((l_enh, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class CrackDataset(Dataset):
    def __init__(self, file_paths, file_labels, img_size=256, train=True):
        self.file_paths = file_paths
        self.file_labels = file_labels
        self.img_size = img_size
        self.train = train
        aug = []
        if train:
            aug = [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                   transforms.RandomRotation(15)]
        self.to_tensor = transforms.Compose(aug + [
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        label = self.file_labels[idx]

        bgr = cv2.imread(img_path)
        if bgr is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        bgr = cv2.resize(bgr, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        bgr = ieap_enhance(bgr)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        from PIL import Image
        pil_img = Image.fromarray(rgb)
        tensor_img = self.to_tensor(pil_img)

        return tensor_img, label, img_path


train_ds = CrackDataset(train_paths, train_labels, IMG_SIZE, train=True)
test_ds = CrackDataset(test_paths, test_labels, IMG_SIZE, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# %% CELL 6 - Model: parallel CNN (ResNet50) + Swin Transformer branches, fused
class DualBranchCrackNet(nn.Module):
    def __init__(self, num_classes=2, fused_channels=512, fused_size=16, img_size=256):
        super().__init__()
        self.fused_size = fused_size

        # ---------------- CNN branch: pretrained ResNet50 ----------------
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        # Use stem + layer1 + layer2 + layer3 (mid-level features, drop layer4+fc)
        self.cnn_stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        cnn_out_channels = 1024  # ResNet50 layer3 output channels

        # ---------------- Swin Transformer branch ----------------
        # NOTE: swin_tiny_patch4_window7_224 was pretrained at 224x224 and by
        # default hard-asserts input size == 224. Passing img_size=256 here
        # tells timm to reconfigure patch_embed for our actual input size.
        # Swin-Tiny uses relative position bias (no absolute pos embedding),
        # so this is a safe, timm-supported override -- not a hack.
        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            features_only=True,
            out_indices=(2,),   # third stage feature map (mid-level, matches CNN depth)
            img_size=img_size,
        )
        swin_out_channels = self.swin.feature_info.channels()[0]

        # ---------------- Projection heads to a common channel dim ----------------
        self.cnn_proj = nn.Conv2d(cnn_out_channels, fused_channels, kernel_size=1)
        self.swin_proj = nn.Conv2d(swin_out_channels, fused_channels, kernel_size=1)

        # ---------------- Confidence Score Head (softmax, trainable) ----------------
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(fused_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        # 1x1 conv used to derive the sigmoid probability heatmap from fused
        # features (CAM-style; not separately supervised -- see notes at top).
        self.heatmap_conv = nn.Conv2d(fused_channels, 1, kernel_size=1)

    def forward(self, x):
        # Same preprocessed tensor is fed to both branches.
        f_cnn = self.cnn_stem(x)
        f_cnn = self.cnn_proj(f_cnn)
        f_cnn = F.interpolate(f_cnn, size=(self.fused_size, self.fused_size),
                               mode="bilinear", align_corners=False)

        f_swin = self.swin(x)[0]
        if f_swin.shape[1] != self.swin_proj.in_channels:
            # timm can return NHWC for some swin variants/versions -- normalize to NCHW.
            f_swin = f_swin.permute(0, 3, 1, 2).contiguous()
        f_swin = self.swin_proj(f_swin)
        f_swin = F.interpolate(f_swin, size=(self.fused_size, self.fused_size),
                                mode="bilinear", align_corners=False)

        fused = f_cnn + f_swin  # F_fused = F_cnn + F_swin

        logits = self.classifier(fused)
        heatmap = torch.sigmoid(self.heatmap_conv(fused))
        heatmap_full = F.interpolate(heatmap, size=(256, 256), mode="bilinear", align_corners=False)

        return logits, heatmap_full, fused


model = DualBranchCrackNet(num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(DEVICE)

# %% CELL 7 - Sanity check BEFORE full training (catches shape/timm-version issues early)
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    logits, heatmap, fused = model(dummy)
    print("logits shape :", logits.shape)     # expect [2, 2]
    print("heatmap shape:", heatmap.shape)    # expect [2, 1, 256, 256]
    print("fused shape  :", fused.shape)      # expect [2, 512, 16, 16]

# %% CELL 8 - Training loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_acc = 0.0
best_state = None
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, lbls, _ in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        logits, _, _ = model(imgs)
        loss = criterion(logits, lbls)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    # ---- validate on the held-out 30% test split each epoch ----
    model.eval()
    val_correct, val_total, val_loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for imgs, lbls, _ in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            logits, _, _ = model(imgs)
            loss = criterion(logits, lbls)
            val_loss_sum += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == lbls).sum().item()
            val_total += lbls.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

print(f"Best val accuracy on 30% holdout: {best_acc:.4f}")

# Save best checkpoint + training history
ckpt_path = os.path.join(OUTPUT_DIR, "crackdualnet_best.pth")
torch.save(best_state, ckpt_path)
pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
print("Saved checkpoint to:", ckpt_path)

# %% CELL 9 - Full evaluation report on the 30% holdout test set (using best weights)
model.load_state_dict(best_state)
model.eval()

all_preds, all_labels, all_paths = [], [], []
with torch.no_grad():
    for imgs, lbls, paths_batch in test_loader:
        imgs = imgs.to(DEVICE)
        logits, _, _ = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(lbls.numpy().tolist())
        all_paths.extend(paths_batch)

acc = accuracy_score(all_labels, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")
cm = confusion_matrix(all_labels, all_preds)

print(f"Test Accuracy : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-score      : {f1:.4f}")
print("Confusion matrix:\n", cm)
print(classification_report(all_labels, all_preds, target_names=list(class_to_idx.keys())))

# %% CELL 10 - Post-processing utilities (Otsu, morphological skeleton, EDT, density)
# These run on the model's sigmoid heatmap output. They are classical CV
# operations (not learned), matching the right-hand side of your diagram.
from skimage.morphology import skeletonize
from scipy.ndimage import distance_transform_edt

def postprocess_heatmap(heatmap_2d_uint8):
    """
    heatmap_2d_uint8: single-channel heatmap, dtype uint8, values 0-255.
    Returns: otsu_thresh, binary_mask, skeleton, edt_map, crack_density_pct
    """
    otsu_thresh, binary_mask = cv2.threshold(
        heatmap_2d_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    bool_mask = binary_mask > 0
    skeleton = skeletonize(bool_mask)
    edt_map = distance_transform_edt(bool_mask)
    crack_density_pct = 100.0 * bool_mask.sum() / bool_mask.size
    return otsu_thresh, binary_mask, skeleton, edt_map, crack_density_pct


def run_full_pipeline_on_image(model, img_path, device):
    """Runs the trained model + post-processing on a single image and
    returns everything needed to reproduce every box in the diagram."""
    bgr = cv2.imread(img_path)
    bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    enhanced = ieap_enhance(bgr)
    rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)

    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    from PIL import Image
    tensor_img = tfm(Image.fromarray(rgb)).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits, heatmap, _ = model(tensor_img)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        heatmap_np = (heatmap.squeeze().cpu().numpy() * 255).astype(np.uint8)

    otsu_thresh, binary_mask, skeleton, edt_map, density = postprocess_heatmap(heatmap_np)

    return {
        "class_probs": probs,               # Confidence Score Head output
        "heatmap": heatmap_np,               # Sigmoid Probability Map
        "otsu_threshold": otsu_thresh,       # Otsu Threshold Head
        "binary_mask": binary_mask,          # Segmented Image
        "skeleton": skeleton,                # Morphological Skeleton
        "edt_map": edt_map,                  # EDT Map
        "crack_density_pct": density,        # Crack Surface Density
    }

# Example usage on one test image:
example_result = run_full_pipeline_on_image(model, test_paths[0], DEVICE)
print("Class probabilities [class0, class1]:", example_result["class_probs"])
print("Otsu threshold:", example_result["otsu_threshold"])
print(f"Crack surface density: {example_result['crack_density_pct']:.2f}%")

# %% CELL 11 - Download these from Kaggle's Output panel when the notebook finishes:
#   /kaggle/working/crackdualnet_best.pth      -> trained model weights
#   /kaggle/working/train_test_split.csv       -> exact 70/30 file split (needed locally)
#   /kaggle/working/training_history.csv       -> loss/accuracy curves
#   /kaggle/working/dataset_meta.json          -> auto-detected path + class mapping (needed locally)

Device: cuda
Auto-detected DATA_DIR: /kaggle/input/datasets/dinjbu/asphalt-cracked-and-uncracked-image-dataset-720/Cracked and Uncracked Image Database_720x720
Classes found: {'Cracked720': 0, 'Uncracked720': 1}
Total images: 3398
Train: 2378 | Test (30% holdout): 1020
Saved split manifest to: /kaggle/working/train_test_split.csv
Download this CSV along with the model checkpoint -- you need it locally to reproduce the exact same 30% test set.
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

logits shape : torch.Size([2, 2])
heatmap shape: torch.Size([2, 1, 256, 256])
fused shape  : torch.Size([2, 512, 16, 16])
Epoch 1/25 | train_loss=0.1143 train_acc=0.9523 | val_loss=0.0181 val_acc=0.9941
Epoch 2/25 | train_loss=0.0626 train_acc=0.9844 | val_loss=0.0328 val_acc=0.9931
Epoch 3/25 | train_loss=0.0179 train_acc=0.9941 | val_loss=0.0063 val_acc=0.9990
Epoch 4/25 | train_loss=0.0124 train_acc=0.9975 | val_loss=0.0064 val_acc=0.9980
Epoch 5/25 | train_loss=0.0095 train_acc=0.9962 | val_loss=0.0084 val_acc=0.9990
Epoch 6/25 | train_loss=0.0189 train_acc=0.9941 | val_loss=0.0029 val_acc=0.9990
Epoch 7/25 | train_loss=0.0064 train_acc=0.9975 | val_loss=0.0058 val_acc=0.9990
Epoch 8/25 | train_loss=0.0166 train_acc=0.9970 | val_loss=0.0118 val_acc=0.9941
Epoch 9/25 | train_loss=0.0134 train_acc=0.9958 | val_loss=0.0079 val_acc=0.9980
Epoch 10/25 | train_loss=0.0030 train_acc=0.9992 | val_loss=0.0040 val_acc=0.9990
Epoch 11/25 | train_loss=0.0016 train_acc=0.9996 | val_loss=0.0127 